## 1. Preparation

In [1]:
from pathlib import Path

# Dataset name
DATASET_NAME = "pcd_merge_kfcv"

# Dataset directory
DATASET = Path(f"/workspace/datasets") / DATASET_NAME

### 1.A Dataset loading

### 1.B Runtime environment settings

In [2]:
# ===== Directory references setup

TRAIN_DIR = Path("/workspace/models")
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# YOLOv11 training directory
TRAIN_DIR_YOLO = TRAIN_DIR / "yolov11"
TRAIN_DIR_YOLO.mkdir(parents=True, exist_ok=True)

In [3]:
# Print GPU information
!nvidia-smi

Sat Jul  4 12:35:15 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA A100-SXM4-80GB          On  | 00000000:BD:00.0 Off |                    0 |
| N/A   41C    P0              62W / 500W |      0MiB / 81920MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

## 2. Training utilities

### 2.A Training parameters

#### 2.A.1 YOLO import and enum definition

In [4]:
# ===== YOLO Import
!apt-get update
!apt-get install -y libgl1
!apt-get install -y libglib2.0-0
!pip install -U ultralytics
!pip install pillow-heif
!pip install -U albumentations

from ultralytics import YOLO

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libgl1 is already the newest version (1.4.0-1).
0 upgraded, 0 newly installed, 0 to remove and 115 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libglib2.0-0 is already the newest version (2.72.4-0ubuntu2.9).
0 upgraded, 0 newly installed, 0 to remove and 115 not upgraded.


In [5]:
# ===== YoloVariant Enum definition

from enum import Enum
class YoloVariant(Enum):
    YOLOv11n = "yolo11n.pt"
    YOLOv11s = "yolo11s.pt"
    YOLOv11m = "yolo11m.pt"
    YOLOv11l = "yolo11l.pt"

#### 2.A.2 Training Configuration class (*YOLOv11TrainConfig*)
Training parameters for YOLO. These are just the parameters of the function `model.train()`. 


Here are defined all the data augmentation introduced to the dataset:

##### Geometric Augmentations

* **`degrees` (30.0):** Rotates the input images randomly within a range of $[-30.0, 30.0]$ degrees.
* **`translate` (0.2):** Translates the images horizontally and vertically by a random fraction within $[-0.2, 0.2]$ of the total image dimensions.
* **`scale` (0.85):** Scales the image dimensions by a random factor within the range $[0.85, 1.15]$.
* **`shear` (10.0):** Shears the image along the horizontal and vertical axes by a random angle within the range of $[-10.0, 10.0]$ degrees.
* **`perspective` (0.002):** Applies a random 3D perspective distortion utilizing a distortion coefficient of $0.002$.

---

##### Photometric Augmentations

* **`hsv_h` (0.005):** Adjusts the image hue channel in the HSV color space by a random fraction within $[-0.005, 0.005]$.
* **`hsv_s` (0.4):** Adjusts the image saturation channel in the HSV color space by a random fraction within $[-0.4, 0.4]$.
* **`hsv_v` (0.75):** Adjusts the image value (brightness) channel in the HSV color space by a random fraction within $[-0.75, 0.75]$.

---

##### Composition Augmentations

* **`mosaic` (1.0):** Sets the execution probability of the mosaic augmentation to $1.0$ ($100\%$), which combines four training images into a single composite layout.
* **`mixup` (0.2):** Sets the execution probability of the mixup augmentation to $0.2$ ($20\%$), which blends two distinct images together with a random alpha transparency parameter.
* **`cutmix` (0.2):** Sets the execution probability of the cutmix augmentation to $0.2$ ($20\%$), which cuts a rectangular patch from one training image and pastes it onto another.

---

##### Custom Albumentations: Blur (`custom_augment`)

This custom block executes an external preprocessing sequence containing controlled blur transformations with an execution probability of $0.8$ ($80\%$).

* **Light Gaussian Blur** (50% selection probability):
    * **`blur_limit` (3, 3):** Sets the Gaussian kernel size to a fixed $3 \times 3$ pixels.
    * **`sigma_limit` (0.2, 0.5):** Samples the Gaussian kernel standard deviation ($\sigma$) from the range $[0.2, 0.5]$.

* **Medium Gaussian Blur** (25% selection probability):
    * **`blur_limit` (3, 5):** Samples the kernel size from the odd integers in the range $[3, 5]$ (resulting in either a $3 \times 3$ or a $5 \times 5$ pixel kernel).
    * **`sigma_limit` (0.6, 1.0):** Samples the Gaussian kernel standard deviation ($\sigma$) from the range $[0.6, 1.0]$.

* **Motion Blur** (25% selection probability):
    * **`blur_limit` (3, 5):** Samples the linear motion blur kernel size from the odd integers in the range $[3, 5]$ (resulting in a kernel length of either 3 or 5 pixels).

##### 2.A.2.A Custom augmentation
Introduced in order to have blurs. This was done as synthetic images have perfect borders (the card in our datasets where cut from some video or generated via Blender and pasted onto a random background), which would lead the model to learn upon those irrealistic borders

In [6]:
# ===== Custom albumentation to introduce blur
import albumentations


custom_augment = albumentations.Compose([

    # 80% probability of having one blur
    albumentations.OneOf([

        # 50% prob of light Gaussian blur
        albumentations.GaussianBlur(
            blur_limit=(3, 3),
            sigma_limit=(0.2, 0.5)
        ),

        # 25% prob of medium Gaussian blur
        albumentations.GaussianBlur(
            blur_limit=(3, 5),
            sigma_limit=(0.6, 1.0)
        ),
    
        # 25% prob of motion nlur    
        albumentations.MotionBlur(
            blur_limit=(3, 5)
        ),
        
    ], p=0.8)
])

##### 2.A.2.B YOLOv11TrainConfig
Training parameters for YOLO. These are just the parameters of the function `model.train()`

In [7]:
# ===== YOLOv11TrainConfig dataclass

from dataclasses import dataclass

@dataclass
class YOLOv11TrainConfig:

    # Path to yaml file
    data: Path = "_to_specify"

    ###  Training hyperparameters

    # Training epochs
    epochs: int = 150

    # Batch size (-1 for auto)
    batch: int = 256

    # Image size (pixels)
    imgsz: int = 512

    # Device to use for training (0 for GPU)
    device: str | int = 0

    # dataloader workers
    workers: int = 8

    # early stopping patience
    patience: int = 20

    # auto mixed precision
    amp: bool = True


    ### Geometric augmentations

    # Rotation (+/- degrees)
    degrees: float = 30.0

    # translation fraction
    translate: float = 0.2

    # scaling factor
    scale: float = 0.85

    # Shear degrees
    shear: float = 10.0

    # Perspective distortion
    perspective: float = 0.002


    ### Photometric augmentations

    # Hue shift
    hsv_h: float = 0.005

    # Saturation gain
    hsv_s: float = 0.4

    # Brightness gain
    hsv_v: float = 0.75


    ### Composition augmentations

    # probability of mosaic
    mosaic: float = 1.0

    # probability of mixup
    mixup: float = 0.2

    # probability of copy-paste of apart of an image
    cutmix: float = 0.2
    

    ### Blur
    augmentations: albumentations.Compose = custom_augment

    
    # Output folder name
    output_dir: str = "dir_to_specify"

##### 2.A.2.C Defaults for YOLOv11 variants

In [8]:
# ===== Defaults for YOLOv11 variants

YOLOv11nTrainConfig_Default = YOLOv11TrainConfig()
YOLOv11sTrainConfig_Default = YOLOv11TrainConfig()
YOLOv11mTrainConfig_Default = YOLOv11TrainConfig()
YOLOv11lTrainConfig_Default = YOLOv11TrainConfig()

#### 2.A.3 Function to instatiate a custom training configuration
Function created to be able to create some configurations starting from the default ones, without having to define many default configurations or manually set all values

In [9]:
# ===== function instantiate_yolo_train_config

from typing import Union

def instantiate_yolo_train_config(yolo_variant: YoloVariant, **overrides):
    # get default config

    if yolo_variant == YoloVariant.YOLOv11n:
        base = YOLOv11nTrainConfig_Default
    elif yolo_variant == YoloVariant.YOLOv11s:
        base = YOLOv11sTrainConfig_Default
    elif yolo_variant == YoloVariant.YOLOv11m:
        base = YOLOv11mTrainConfig_Default
    elif yolo_variant == YoloVariant.YOLOv11l:
        base = YOLOv11lTrainConfig_Default
    else:
        raise ValueError(f"Unsupported YOLO variant: {yolo_variant}")

    # merge defaults with overrides
    config_dict = {**base.__dict__, **overrides}

    # force to path
    if isinstance(config_dict["data"], str):
        config_dict["data"] = Path(config_dict["data"])

    return YOLOv11TrainConfig(**config_dict)

### 2.B Training functions

#### 2.B.1 Model evaluation

##### 2.B.1.A Output of model evaluation

We used this following code to discover all available fields from the `metrics` objects (the result of `model.val()`):
```python
def dump(obj, indent=0):
    for k, v in obj.__dict__.items():
        print("    " * indent + str(k))
        if hasattr(v, "__dict__"):
            dump(v, indent + 1)
```

The structure found was:
```python

"""
        "conf_values":metrics.box.px,
        "p_curve":    metrics.box.p_curve,
        "r_curve":    metrics.box.r_curve,
        "f1_curve":   metrics.box.f1_curve,
        "f1_curve":   metrics.box.prec_values
"""

# Class names, indexed
metrics.names

# Directory where results are stored
metrics.save_dir

metrics.stats
metrics.nt_per_class
metrics.nt_per_image

# Plotting data for the confusion matrix
metrics.confusion_matrix  
    metrics.confusion_matrix.task
    metrics.confusion_matrix.nc
    metrics.confusion_matrix.matrix # the number of samples for each cell of the confusion matrix
    metrics.confusion_matrix.names
    metrics.confusion_matrix.matches

# Speed data
metrics.speed
    metrics.speed.preprocess
    metrics.speed.inference
    metrics.speed.loss
    metrics.speed.postprocess
    
metrics.box:

    # Precision per class
    metrics.box.p

    # Recall per class
    metrics.box.r

    # F1 per class
    metrics.box.f1

    # AP for each class at 10 points
    metrics.box.all_ap
        metrics.box.all_ap[class_index] = array(10)

    # class index
    metrics.box.ap_class_index

    # Number of classes
    metrics.box.nc
    metrics.box.image_metrics # metrics for every single image
        metrics.box.image_metrics[image_id].precision
        metrics.box.image_metrics[image_id].recall
        metrics.box.image_metrics[image_id].f1
        metrics.box.image_metrics[image_id].tp
        metrics.box.image_metrics[image_id].fp
        metrics.box.image_metrics[image_id].fn
        
    metrics.box.p_curve # plotting data for each class of the P Curve
        metrics.box.p_curve[class_index] = array
        
    metrics.box.r_curve # plotting data for each class of the R Curve
        metrics.box.r_curve[class_index] = array
        
    metrics.box.f1_curve # plotting data for each class of the F1 Curve
        metrics.box.f1_curve[class_index] = array

    # Precision points used (1000 from 0 to 1), with their values
    metrics.box.px

    # Graph data for the global PR curve we believe
    metrics.box.prec_values
```

---

We decided to save only a subset of these information

###### Object to json conversion
Function used to convert the output of `model.val()` to a json object, to be stored in a file

In [10]:
# ===== to_jsonable function

import json
import numpy as np
import torch

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [to_jsonable(x) for x in obj]

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()

    if hasattr(obj, "tolist"):
        return obj.tolist()

    if hasattr(obj, "__dict__"):
        return {k: to_jsonable(v) for k, v in obj.__dict__.items()}

    # numpy scalars
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()

    return obj

##### 2.B.1.B Model evaluation function

In [11]:
# ===== evaluate_model

import json

def evaluate_model(model, train_config: YOLOv11TrainConfig):

    eval_subfolder = train_config.output_dir / 'eval'
    
    # Evaluate the model
    metrics = model.val(
        split='test',
        plots=True,
        save_json=True,
        name=str(eval_subfolder)
    )
    
    results = {
        "all_ap":     metrics.box.all_ap,
        "mAP50-95":   metrics.box.map,
        "mAP50":      metrics.box.map50,
        "mAP75":      metrics.box.map75,
        "mAP50-95_per_class": metrics.box.maps.tolist(),
        "precision":  metrics.box.p.tolist(),
        "recall":     metrics.box.r.tolist(),
        "f1":         metrics.box.f1.tolist(),
        "class_names":list(metrics.names.values()),
        "fitness":    float(metrics.fitness),
        "speed":      metrics.speed,   
        "save_dir":   str(metrics.save_dir)
    }

    results = to_jsonable(results)
    
    results_file = eval_subfolder / 'results.json'
    with open(results_file, 'w') as f:
        json.dump(results, f, indent=2)

#### 2.B.2 Model training

##### 2.B.2.A Single model (`train_yolo`)

In [12]:
def train_yolo(model, train_config: YOLOv11TrainConfig):

    # Train the model
    model.train(
        data=str(train_config.data),
        epochs=train_config.epochs,
        batch=train_config.batch,
        imgsz=train_config.imgsz,
        device=train_config.device,
        workers=train_config.workers,
        patience=train_config.patience,
        amp=train_config.amp,

        degrees=train_config.degrees,
        translate=train_config.translate,
        scale=train_config.scale,
        shear=train_config.shear,
        perspective=train_config.perspective,

        hsv_h=train_config.hsv_h,
        hsv_s=train_config.hsv_s,
        hsv_v=train_config.hsv_v,

        augmentations=train_config.augmentations,

        mosaic=train_config.mosaic,
        mixup=train_config.mixup,
        cutmix=train_config.cutmix,

        name=str(train_config.output_dir)
    )

    # Evaluate the model
    evaluate_model(model, train_config)

##### 2.B.2.B Multiple instances (K-Fold Training) (`train_k_fold`)
This function iterates over the single `train_yolo` and automatically overwrites the `data` and `output_dir` parameters for each fold

In [13]:
YAML_DIR = DATASET / "yamls"

In [14]:
# ===== K-Fold Training
from copy import deepcopy

def train_k_fold(model, train_config: YOLOv11TrainConfig, folds = [0, 1, 2, 3, 4]):
    """
    Train the model over all folds
    """

    # Retrieve all fold yaml files from YAML_DIR
    fold_yaml_files = sorted(YAML_DIR.glob("fold_*.yaml"))

    print(f"Searching in {YAML_DIR}, found:")
    for yf in fold_yaml_files:
        print(yf)

    # Loop over all folds (named data_fold_0.yaml, data_fold_1.yaml, etc)
    for fold_yaml in fold_yaml_files:

        # If fold set is specify, continue if not specified
        fold_index = int(str(fold_yaml).removesuffix(".yaml").split("_")[-1])

        if fold_index not in folds:
            print(f"skipping fold {fold_index}")
            continue


        print(f"Training on fold: {fold_yaml.name}")

        # create isolated config per fold
        cfg = deepcopy(train_config)

        # Update the data path in the training configuration
        cfg.data = fold_yaml

        # Update the output directory to include the fold name (<output_dir>/fold_i/)
        cfg.output_dir = Path(train_config.output_dir) / fold_yaml.stem

        # create isolated model per fold
        model_fold = deepcopy(model)

        # Train the model with the updated configuration
        train_yolo(model_fold, cfg)



## 3. Training

### 3.A YOLOv11 **Nano**

In [15]:
# Load pretrained (pt) YOLOv11n
yolo11n = YOLO("yolo11n.pt")


# ===== Create configuration

# Train directory for nano model
TRAIN_DIR_NANO = TRAIN_DIR_YOLO / "nano"

# Create config
yolo11n_config = instantiate_yolo_train_config(
    YoloVariant.YOLOv11n,
    output_dir=TRAIN_DIR_NANO,
)

print(f"Training YOLOv11n with config: {yolo11n_config}")


# ===== K-Fold training
train_k_fold(yolo11n, yolo11n_config, [])

Training YOLOv11n with config: YOLOv11TrainConfig(data=PosixPath('_to_specify'), epochs=150, batch=256, imgsz=512, device=0, workers=8, patience=20, amp=True, degrees=30.0, translate=0.2, scale=0.85, shear=10.0, perspective=0.002, hsv_h=0.005, hsv_s=0.4, hsv_v=0.75, mosaic=1.0, mixup=0.2, cutmix=0.2, augmentations=Compose([
  OneOf([
    GaussianBlur(p=0.5, blur_limit=(3, 3), sigma_limit=(0.2, 0.5)),
    GaussianBlur(p=0.5, blur_limit=(3, 5), sigma_limit=(0.6, 1.0)),
    MotionBlur(p=0.5, allow_shifted=True, angle_range=(0.0, 360.0), blur_limit=(3, 5), direction_range=(-1.0, 1.0)),
  ], p=0.8),
], p=1.0, bbox_params=None, keypoint_params=None, additional_targets={}, is_check_shapes=True), output_dir=PosixPath('/workspace/models_rerun/yolov11/nano'))
Searching in /workspace/datasets/pcd_merge_kfcv/yamls, found:
/workspace/datasets/pcd_merge_kfcv/yamls/fold_0.yaml
/workspace/datasets/pcd_merge_kfcv/yamls/fold_1.yaml
/workspace/datasets/pcd_merge_kfcv/yamls/fold_2.yaml
/workspace/datasets

### 3.B YOLOv11 **Small**

In [16]:
# Load pretrained (pt) YOLOv11s
yolo11s = YOLO("yolo11s.pt")


# ===== Create configuration

# Train directory for nano model
TRAIN_DIR_SMALL = TRAIN_DIR_YOLO / "small"

yolo11s_config = instantiate_yolo_train_config(
    YoloVariant.YOLOv11s,
    output_dir=TRAIN_DIR_SMALL,
)

print(f"Training YOLOv11s with config: {yolo11s_config}")


# ===== K-Fold training
train_k_fold(yolo11s, yolo11s_config, [])

Training YOLOv11s with config: YOLOv11TrainConfig(data=PosixPath('_to_specify'), epochs=150, batch=256, imgsz=512, device=0, workers=8, patience=20, amp=True, degrees=30.0, translate=0.2, scale=0.85, shear=10.0, perspective=0.002, hsv_h=0.005, hsv_s=0.4, hsv_v=0.75, mosaic=1.0, mixup=0.2, cutmix=0.2, augmentations=Compose([
  OneOf([
    GaussianBlur(p=0.5, blur_limit=(3, 3), sigma_limit=(0.2, 0.5)),
    GaussianBlur(p=0.5, blur_limit=(3, 5), sigma_limit=(0.6, 1.0)),
    MotionBlur(p=0.5, allow_shifted=True, angle_range=(0.0, 360.0), blur_limit=(3, 5), direction_range=(-1.0, 1.0)),
  ], p=0.8),
], p=1.0, bbox_params=None, keypoint_params=None, additional_targets={}, is_check_shapes=True), output_dir=PosixPath('/workspace/models_rerun/yolov11/small'))
Searching in /workspace/datasets/pcd_merge_kfcv/yamls, found:
/workspace/datasets/pcd_merge_kfcv/yamls/fold_0.yaml
/workspace/datasets/pcd_merge_kfcv/yamls/fold_1.yaml
/workspace/datasets/pcd_merge_kfcv/yamls/fold_2.yaml
/workspace/dataset

### 3.C YOLOv11 **Medium**

In [ ]:
# Load pretrained (pt) YOLOv11m
yolo11m = YOLO("yolo11m.pt")


# ===== Create configuration

# Train directory for nano model
TRAIN_DIR_MEDIUM = TRAIN_DIR_YOLO / "medium"

yolo11m_config = instantiate_yolo_train_config(
    YoloVariant.YOLOv11m,
    output_dir=TRAIN_DIR_MEDIUM,
)

print(f"Training YOLOv11m with config: {yolo11m_config}")


# ===== K-Fold training
train_k_fold(yolo11m, yolo11m_config, [])

Training YOLOv11m with config: YOLOv11TrainConfig(data=PosixPath('_to_specify'), epochs=150, batch=256, imgsz=512, device=0, workers=8, patience=30, amp=True, degrees=30.0, translate=0.2, scale=0.85, shear=10.0, perspective=0.002, hsv_h=0.005, hsv_s=0.4, hsv_v=0.75, mosaic=1.0, mixup=0.2, cutmix=0.2, augmentations=Compose([
  OneOf([
    GaussianBlur(p=0.5, blur_limit=(3, 3), sigma_limit=(0.2, 0.5)),
    GaussianBlur(p=0.5, blur_limit=(3, 5), sigma_limit=(0.6, 1.0)),
    MotionBlur(p=0.5, allow_shifted=True, angle_range=(0.0, 360.0), blur_limit=(3, 5), direction_range=(-1.0, 1.0)),
  ], p=0.8),
], p=1.0, bbox_params=None, keypoint_params=None, additional_targets={}, is_check_shapes=True), output_dir=PosixPath('/workspace/models_rerun/yolov11/medium'))
Searching in /workspace/datasets/pcd_merge_kfcv/yamls, found:
/workspace/datasets/pcd_merge_kfcv/yamls/fold_0.yaml
/workspace/datasets/pcd_merge_kfcv/yamls/fold_1.yaml
/workspace/datasets/pcd_merge_kfcv/yamls/fold_2.yaml
/workspace/datase

/usr/local/lib/python3.10/dist-packages/albumentations/core/composition.py:465: UserWarning: transforms is single transform, but a sequence is expected! Transform will be wrapped into list.
  super().__init__(


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1806.0±1005.4 MB/s, size: 51.4 KB)
val: Scanning /workspace/datasets/pcd_merge_kfcv/labels... 4816 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4816/4816 2.8Kit/s 1.7s0.1s
val: New cache created: /workspace/datasets/pcd_merge_kfcv/labels.cache
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: MuSGD(lr=0.01, momentum=0.9) with parameter groups 106 weight(decay=0.0), 113 weight(decay=0.002), 112 bias(decay=0.0)
Plotting labels to /workspace/models_rerun/yolov11/medium/fold_1-2/labels.jpg... 
Image sizes 512 train, 512 val
Using 8 dataloader workers
Logging results to /workspace/models_rerun/yolov11/medium/fold_1-2
Starting training for 150 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/150      78.1G      2.012      3.888      1.426        455        512: 100% ━━━━━━━━━━━━ 7

/usr/local/lib/python3.10/dist-packages/ultralytics/data/augment.py:1255: RuntimeWarning: divide by zero encountered in divide
  xy = (xy[:, :2] / xy[:, 2:3] if self.perspective else xy[:, :2]).reshape(n, 8)  # perspective rescale or affine


      3/150      77.7G      1.526      1.708      1.071        448        512: 100% ━━━━━━━━━━━━ 76/76 1.1s/it 1:260.8sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 3.1s/it 30.6s.2ss
                   all       4816      18531       0.59       0.58      0.609      0.388

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/150      77.8G       1.48      1.572      1.066        439        512: 100% ━━━━━━━━━━━━ 76/76 1.1s/it 1:260.8sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.3s/it 13.1s.3s
                   all       4816      18531        0.8      0.858      0.887      0.519

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      5/150      77.8G      1.413      1.411      1.045        518        512: 100% ━━━━━━━━━━━━ 76/76 1.1s/it 1:260.8sss
                 Class     Images  

...

      6/150      77.8G      1.347      1.289      1.026        520        512: 100% ━━━━━━━━━━━━ 76/76 1.1s/it 1:260.8sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.3s/it 13.0s.3s
                   all       4816      18531      0.926      0.962      0.973      0.734

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      7/150      77.8G      1.307      1.214      1.013        550        512: 100% ━━━━━━━━━━━━ 76/76 1.1s/it 1:260.8sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.3s/it 12.9s.3s
                   all       4816      18531      0.924      0.966      0.969      0.655

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      8/150      77.7G      1.263       1.15      1.001        490        512: 100% ━━━━━━━━━━━━ 76/76 1.1s/it 1:260.8sss
                 Class     Images  I

/usr/local/lib/python3.10/dist-packages/ultralytics/data/augment.py:1255: RuntimeWarning: divide by zero encountered in divide
  xy = (xy[:, :2] / xy[:, 2:3] if self.perspective else xy[:, :2]).reshape(n, 8)  # perspective rescale or affine


     20/150      77.6G     0.9706     0.8453     0.9256        532        512: 100% ━━━━━━━━━━━━ 76/76 1.1s/it 1:260.8ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.2s/it 12.4s.3s
                   all       4816      18531      0.955      0.991      0.985      0.836

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     21/150      77.2G     0.9546     0.8364     0.9209        475        512: 100% ━━━━━━━━━━━━ 76/76 1.1s/it 1:260.8sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.2s/it 12.2s.2s
                   all       4816      18531      0.958      0.988      0.986      0.765

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     22/150      77.8G     0.9501      0.826     0.9213        479        512: 100% ━━━━━━━━━━━━ 76/76 1.1s/it 1:260.8sss
                 Class     Images  In

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.3s/it 12.6s.3s
                   all       4816      18531      0.976      0.984       0.99      0.906

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
     92/150      77.8G     0.6745     0.5886     0.8549       1935        512: 86% ━━━━━━━━━━── 66/76 1.1s/it 1:16<11.4ss

### 3.D YOLOv11 **Large**

In [ ]:
# Load pretrained (pt) YOLOv11l
yolo11l = YOLO("yolo11l.pt")


# ===== Create configuration

# Train directory for large model
TRAIN_DIR_LARGE = TRAIN_DIR_YOLO / "large"

yolo11l_config = instantiate_yolo_train_config(
    YoloVariant.YOLOv11l,
    output_dir=TRAIN_DIR_LARGE,
)

print(f"Training YOLOv11m with config: {yolo11l_config}")


# ===== K-Fold training
train_k_fold(yolo11l, yolo11l_config, [])